# When you only have function values

Often there are no gradients: only a data set of input points `X` and outputs `f`. This
tutorial estimates subspaces from such a data set with four methods and compares them:

- **OLS**: the gradient of a global linear fit;
- **QPHD**: the gradients of a global quadratic fit;
- **SIR**: sliced inverse regression (Li 1991);
- **local linear gradients**: gradients estimated from nearby points, then the usual
  active subspace.

{doc}`../user_guide/discovering` lists every estimator and its options,
{doc}`../theory/estimators` states what each one estimates, and
{doc}`../theory/estimator_selection` helps choose one.

## A data set with a known answer

The data are synthetic so that the answer is known. Six inputs are Gaussian with the
means and variances below (physical units). With $x$ the normalized inputs,

$$f = \exp(0.7\,a^\top x) + \tfrac12 (b^\top x)^2$$

for two orthogonal unit vectors $a$ and $b$. So $f$ depends on $x$ only through
$a^\top x$ and $b^\top x$, and the true active subspace is the plane
$\operatorname{span}\lbrace a, b\rbrace$. The second term is even in $b^\top x$, which
matters below.

In [ ]:
%matplotlib inline
import warnings

import numpy as np

import active_subspaces as ac

space = ac.Gaussian(
    mean=[1.0, -2.0, 0.0, 5.0, 10.0, 0.0],
    cov=[0.25, 4.0, 1.0, 0.01, 100.0, 9.0],  # variances
    names=("p1", "p2", "p3", "p4", "p5", "p6"),
)
a = np.array([1.0, 2.0, 0.0, 0.0, -1.0, 0.5])
a /= np.linalg.norm(a)
b = np.array([0.0, 0.0, 1.0, 1.0, 0.0, -1.0])
b /= np.linalg.norm(b)

rng = np.random.default_rng(7)
X = space.sample(400, rng=rng)  # physical inputs, (400, 6)
x = space.to_normalized(X)  # used only to define the synthetic outputs
f = np.exp(0.7 * (x @ a)) + 0.5 * (x @ b) ** 2
X.shape, f.shape

From here on, the analysis uses only `X`, `f` and `space`, as it would with real data.
The subspaces are compared with the projector distance
$\operatorname{dist}(U, V) = \lVert UU^\top - VV^\top \rVert_2$, the sine of the
largest principal angle between the spans of $U$ and $V$: 0 for equal subspaces, 1 when
some direction of one is orthogonal to the other.

In [ ]:
def dist(U, V):
    """Projector distance between the spans of U and V (sine of the largest principal angle)."""
    U = np.linalg.qr(np.reshape(U, (len(U), -1)))[0]
    V = np.linalg.qr(np.reshape(V, (len(V), -1)))[0]
    return np.linalg.norm(U @ U.T - V @ V.T, 2)

## OLS: a global linear fit

`estimator="ols"` fits $f \approx c + g^\top x$ by least squares and returns the
direction of $g$. It finds at most one direction. For Gaussian inputs $g$ estimates the
average gradient, which here points along $a$: the quadratic term in $b^\top x$
averages out.

In [ ]:
ols = ac.active_subspace(X=X, f=f, space=space, estimator="ols")
print("n =", ols.n, "| eigenvalues:", np.round(ols.eigenvalues, 3))
print(f"dist(w_1, a) = {dist(ols.W1, a):.3f}")

## QPHD: a global quadratic fit

`estimator="qphd"` fits $f \approx c + g^\top x + \tfrac12 x^\top H x$ by least squares
and returns the eigenvectors of the average outer product of the fitted gradient,
$\hat C = g g^\top + H \Sigma H$, where $\Sigma$ is the covariance of $x$ ($I$ here). This
follows from Corollary 3.1 of Li (1992) with $q = 2$. It needs at least
$(m+1)(m+2)/2 = 28$ points here and sees both directions. A bootstrap works as for
gradients: it resamples the rows of `X` and `f`.

In [ ]:
qphd = ac.active_subspace(X=X, f=f, space=space, estimator="qphd", nboot=100, rng=8)
print("n =", qphd.n, "| eigenvalues:", np.round(qphd.eigenvalues, 3))
print(f"dist(W_1, span{{a, b}}) = {dist(qphd.W1, np.column_stack([a, b])):.3f}")
fig, ax = ac.plotting.plot_eigenvalues(qphd)
fig

## SIR: sliced inverse regression

SIR sorts the data by $f$, cuts them into slices (10 by default) and looks at how the
mean of $x$ moves from slice to slice. Where $f$ is even in a direction, as in $b$, the
slice means do not move along it, so SIR cannot see $b$ (Li 1991).

**Departure from Li (1991).** SIR, pHd and SAVE whiten the normalized inputs with
$\Sigma^{-1/2}$, where $\Sigma$ is the covariance of the declared input space ($I$ for a
Gaussian space, $\tfrac13 I$ for a uniform one), not with the sample covariance
$\hat\Sigma$ that the published method uses.

In [ ]:
sir = ac.active_subspace(X=X, f=f, space=space, estimator="sir")
print("n =", sir.n, "| eigenvalues:", np.round(sir.eigenvalues, 3))
print(f"dist(w_1, a) = {dist(sir.W1, a):.3f}")

## Local linear gradients

`ac.local_linear_gradients` fits a linear model to the nearest points around each sample
(in normalized coordinates) and takes its slope as the gradient there. The result is an
`ac.GradientResult` that carries its space, so it can be passed straight to
`ac.active_subspace`. These gradients are rough, but the active subspace averages over
all of them.

In [ ]:
llg = ac.local_linear_gradients(X=X, f=f, space=space)
print("gradients:", llg.grads.shape, "from", llg.p, "points each")
local = ac.active_subspace(grads=llg, nboot=100, rng=9)
print("n =", local.n, "| eigenvalues:", np.round(local.eigenvalues, 3))

The dimension rule picks $n = 1$ here: $\lambda_1$ is about twice $\lambda_2$, and the
gap after $\lambda_2$ is smaller than the gap after $\lambda_1$. The second eigenvector
is still informative; look at the two-dimensional subspace with `ac.with_dimension`.

## Comparison

One-dimensional estimates are compared with $a$, two-dimensional ones with the plane.

In [ ]:
plane = np.column_stack([a, b])
print(f"{'estimator':<24}{'n':>3}{'dist(w_1, a)':>15}{'dist(W_1, plane), n = 2':>26}")
for label, s in [("OLS", ols), ("SIR", sir), ("QPHD", qphd), ("local linear + AS", local)]:
    two = "      -" if s.estimator == "ols" else f"{dist(s.W[:, :2], plane):7.3f}"
    print(f"{label:<24}{s.n:>3}{dist(s.W[:, :1], a):>15.3f}{two:>26}")
print(f"QPHD vs local linear, n = 2: {dist(qphd.W[:, :2], local.W[:, :2]):.3f}")

OLS and SIR find $a$ but miss $b$; QPHD and the local linear gradients recover the
plane. A two-dimensional sufficient summary plot of the local-linear subspace shows
the structure: $f$ grows exponentially along one axis and quadratically, symmetrically
along the other.

In [ ]:
fig, ax = ac.plotting.plot_sufficient_summary(ac.with_dimension(local, 2), X=X, f=f)
fig

## Uniform inputs: a validity warning

OLS, QPHD, SIR, pHd and SAVE rest on assumptions about the input density, such as
Gaussian inputs. On a uniform space the library still computes the estimate, but issues
an `ac.ValidityWarning` with the code `validity.assumption`. Here is OLS on values of the
wing weight function, whose inputs are uniform:

In [ ]:
wing = ac.test_functions.WING_WEIGHT
Xw = wing.space.sample(200, rng=10)
fw = ac.evaluate(wing, X=Xw)
with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    ols_w = ac.active_subspace(X=Xw, f=fw, space=wing.space, estimator="ols")
for w in caught:
    print(type(w.message).__name__, w.message.code)
Gw = ac.evaluate_grads(wing, X=Xw)
as_w = ac.active_subspace(grads=Gw, space=wing.space)
print(f"dist to the gradient-based w_1: {dist(ols_w.W1, as_w.W1):.3f}")

The OLS direction is still close to the gradient-based one for this nearly monotone
function, but nothing guarantees that in general. {doc}`../user_guide/errors` lists the
warning codes.

## References

- Constantine, P. G. (2015). *Active Subspaces: Emerging Ideas for Dimension Reduction
  in Parameter Studies.* SIAM.
- Li, K.-C. (1991). Sliced inverse regression for dimension reduction. *Journal of the
  American Statistical Association*, 86(414), 316–327.
- Li, K.-C. (1992). On principal Hessian directions for data visualization and dimension
  reduction: another application of Stein's lemma. *Journal of the American Statistical
  Association*, 87(420), 1025–1039.